# Polars on GPU: TPC-H lineitem (SF1000) from S3

Four queries from easy to hard. Each cell runs the query on the CPU (Polars streaming engine) and on the GPU (`RayEngine`) and prints the time with `%time`.

In [1]:
import os, boto3

creds = boto3.Session().get_credentials().get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"] = creds.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = creds.secret_key
if creds.token:  # instance roles use temporary credentials
    os.environ["AWS_SESSION_TOKEN"] = creds.token
os.environ["AWS_DEFAULT_REGION"] = "us-east-2"  # use the region of the rapids-tpch bucket

In [2]:
# Check the instance can read the bucket (shell commands)
!aws sts get-caller-identity
!aws s3 ls s3://rapids-tpch/scale-1000/lineitem/ | head
!aws s3 ls s3://rapids-tpch/scale-1000/lineitem/ --summarize --human-readable | tail -3

{
    "UserId": "AROAYFLEYUUABISMVCCEW:i-0319a9124edce841a",
    "Account": "561241433344",
    "Arn": "arn:aws:sts::561241433344:assumed-role/etl-cloud-benchmarking-role/i-0319a9124edce841a"
}
2025-09-16 18:26:28  485500272 part.0.parquet
2025-09-16 18:26:28  485507127 part.1.parquet
2025-09-16 18:26:31  485749414 part.10.parquet
2025-09-16 18:26:31  485824711 part.100.parquet
2025-09-16 18:26:34  485828489 part.101.parquet
2025-09-16 18:26:38  485692314 part.102.parquet
2025-09-16 18:26:39  485254846 part.103.parquet
2025-09-16 18:26:47  485699512 part.104.parquet
2025-09-16 18:26:50  485517671 part.105.parquet
2025-09-16 18:26:52  485574602 part.106.parquet

aws: [ERROR]: [Errno 32] Broken pipe
Exception ignored while flushing sys.stdout:
BrokenPipeError: [Errno 32] Broken pipe

Total Objects: 500
   Total Size: 226.1 GiB


In [3]:
import polars as pl
from cudf_polars.engine.ray import RayEngine   # needs: pip install ray

# ~150 GB of lineitem: first 309 files (~485 MB each)
N_FILES = 207
files = [f"s3://rapids-tpch/scale-1000/lineitem/part.{i}.parquet" for i in range(N_FILES)]
lineitem = pl.scan_parquet(files)   # add the same storage_options/args you used for `lineitem`

lineitem.collect_schema()


Schema([('l_orderkey', Int64),
        ('l_partkey', Int64),
        ('l_suppkey', Int64),
        ('l_linenumber', Int64),
        ('l_quantity', Float64),
        ('l_extendedprice', Float64),
        ('l_discount', Float64),
        ('l_tax', Float64),
        ('l_returnflag', String),
        ('l_linestatus', String),
        ('l_shipdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_commitdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_receiptdate', Datetime(time_unit='ms', time_zone=None)),
        ('l_shipinstruct', String),
        ('l_shipmode', String),
        ('l_comment', String)])

# Background

TPC-H models a global wholesale parts distributor, and lineitem is every shipment line it has ever sent. That’s ~2.5 billion lines over seven years (1992 to 1998), ~200 million parts, and ~10 million suppliers. You play the analyst, and each query answers a question the CEO might ask.


# Column

| Column | Meaning |
|---|---|
| `l_orderkey` | The order this line belongs to, which links to the orders table. |
| `l_partkey` | The part that was ordered, which links to part. |
| `l_suppkey` | The supplier that ships it, which links to supplier. |
| `l_linenumber` | The line's position within its order (1 to 7). |
| `l_quantity` | Units ordered (1 to 50). |
| `l_extendedprice` | Quantity × the part's price: the line total before discount and tax. |
| `l_discount` | Discount as a fraction (0.00 to 0.10). |
| `l_tax` | Tax as a fraction (0.00 to 0.08). |
| `l_returnflag` | R = returned, A = accepted, N = not yet received. R and A apply if it was received by the dataset's "today" (June 17, 1995), otherwise N. |
| `l_linestatus` | O = open (shipped after June 17, 1995), F = filled (shipped on or before). |
| `l_shipdate` | Date the line was shipped. |
| `l_commitdate` | Date the supplier promised delivery. |
| `l_receiptdate` | Date the customer received it. |
| `l_shipinstruct` | Delivery instruction: DELIVER IN PERSON, COLLECT COD, NONE, or TAKE BACK RETURN. |
| `l_shipmode` | How it was shipped: AIR, REG AIR, RAIL, SHIP, TRUCK, MAIL, or FOB. |
| `l_comment` | Random filler text. |

In [7]:
lineitem.head().collect()

l_orderkey,l_partkey,l_suppkey,l_linenumber,l_quantity,l_extendedprice,l_discount,l_tax,l_returnflag,l_linestatus,l_shipdate,l_commitdate,l_receiptdate,l_shipinstruct,l_shipmode,l_comment
i64,i64,i64,i64,f64,f64,f64,f64,str,str,datetime[ms],datetime[ms],datetime[ms],str,str,str
1,155189345,7689361,1,17.0,24252.03,0.04,0.02,"""N""","""O""",1996-03-13 00:00:00,1996-02-12 00:00:00,1996-03-22 00:00:00,"""DELIVER IN PERSON""","""TRUCK""","""egular courts above the"""
1,67309080,7309081,2,36.0,39085.92,0.09,0.06,"""N""","""O""",1996-04-12 00:00:00,1996-02-28 00:00:00,1996-04-20 00:00:00,"""TAKE BACK RETURN""","""MAIL""","""ly final dependencies: slyly b…"
1,63699776,3699777,3,8.0,14180.72,0.1,0.02,"""N""","""O""",1996-01-29 00:00:00,1996-03-05 00:00:00,1996-01-31 00:00:00,"""TAKE BACK RETURN""","""REG AIR""","""riously. regular, express dep"""
1,2131495,4631496,4,28.0,42738.92,0.09,0.06,"""N""","""O""",1996-04-21 00:00:00,1996-03-30 00:00:00,1996-05-16 00:00:00,"""NONE""","""AIR""","""lites. fluffily even de"""
1,24026634,1526641,5,24.0,37426.32,0.1,0.04,"""N""","""O""",1996-03-30 00:00:00,1996-03-14 00:00:00,1996-04-01 00:00:00,"""NONE""","""FOB""",""" pending foxes. slyly re"""


In [6]:
# Start the GPU engine once and reuse it
engine = RayEngine()

2026-10-27 23:40:41,904	WARNING authentication_token_setup.py:85 -- Token authentication is enabled for this Ray cluster. Set RAY_AUTH_MODE=disabled to opt out. For more information, see https://docs.ray.io/en/latest/ray-security/token-auth.html
2026-10-27 23:40:45,233	INFO worker.py:2024 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


A parts distributor with 2.5 billion shipment lines. The CEO has four questions, and we’re going to answer them twice: once on CPU, once on GPU.

# Q1. What are our best-selling parts?

We have 200 million products, and we need to total the sales of every one of them. The winner is part No.17,389,964: 698 units, 1.38M. It wasn’t a blockbuster. The top 10 are within about 6% of each other, so our best seller is barely ahead of the pack.


In [6]:
# Query 1 (easy): revenue per part, top 10  (~200M groups)
q1 = (
    lineitem
    .filter(pl.col("l_shipdate") <= pl.date(1998, 9, 2))
    .group_by("l_partkey")
    .agg(
        pl.col("l_quantity").sum().alias("sum_qty"),
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum().alias("revenue"),
    )
    .sort("revenue", descending=True)
    .head(10)
)

%time cpu = q1.collect(engine="streaming")
%time gpu = q1.collect(engine=engine)
gpu


CPU times: user 11min 5s, sys: 1min 42s, total: 12min 47s
Wall time: 40.1 s
CPU times: user 202 ms, sys: 2.34 s, total: 2.54 s
Wall time: 21.6 s


l_partkey,sum_qty,revenue
i64,f64,f64
17389964,698.0,1.3805e6
186981992,677.0,1.3473e6
128747975,714.0,1.3461e6
142572937,695.0,1.3440e6
54126955,699.0,1.3217e6
50382999,667.0,1.3203e6
73398981,663.0,1.3106e6
115965910,691.0,1.3099e6
99581958,670.0,1.3076e6


 CPU used 12 minutes 47 seconds of processor time, and the GPU used 2.5 seconds.

# Q2. Which suppliers earn the most, and how do they ship?

Now we’re looking at 70 million supplier and shipping-method combinations. The top earner is supplier No.1,160,979 shipping by truck: 4.82M from 92 lines. Six of the seven shipping modes show up in the top 10, so there’s no magic mode. FOB appears three times and AIR once.

In [7]:
# Query 2 (medium): revenue per supplier and ship mode, ALL rows (no date filter)
q2 = (
    lineitem
    .group_by(["l_suppkey", "l_shipmode"])      # 2-column key, ~70M groups
    .agg(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum().alias("revenue"),
        pl.col("l_quantity").mean().alias("avg_qty"),
        pl.len().alias("n_lines"),
    )
    .sort("revenue", descending=True)
    .head(10)
)

%time cpu = q2.collect(engine="streaming")
%time gpu = q2.collect(engine=engine)
gpu


CPU times: user 15min 48s, sys: 2min 24s, total: 18min 12s
Wall time: 37.5 s
CPU times: user 152 ms, sys: 2.62 s, total: 2.77 s
Wall time: 18.7 s


l_suppkey,l_shipmode,revenue,avg_qty,n_lines
i64,str,f64,f64,u64
1160979,"""TRUCK""",4.8202e6,28.456522,92
7491995,"""REG AIR""",4.7263e6,29.841463,82
7264982,"""FOB""",4.7085e6,26.457447,94
6526986,"""MAIL""",4.6658e6,30.011765,85
5078924,"""FOB""",4.6538e6,30.457831,83
4072960,"""REG AIR""",4.6527e6,29.082353,85
7179984,"""FOB""",4.6338e6,24.683168,101
3810958,"""AIR""",4.6219e6,29.068182,88
3686999,"""MAIL""",4.6100e6,29.493827,81


The CPU used 18 minutes of processor time, against 2.8 seconds for the GPU.

# Q3. Which part and supplier relationships matter most?
This is the supply chain itself, about 800 million relationships. Each of the top 10 earns $1.26 to 1.30M from only 21 to 23 lines, while the typical relationship has about three. The big earners are a handful of relationships with unusually heavy traffic.

In [8]:
# Query 3 (harder): revenue per part AND supplier, all rows  (~800M groups, memory-heavy on CPU)
q3 = (
    lineitem
    .group_by(["l_partkey", "l_suppkey"])
    .agg(
        (pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum().alias("revenue"),
        pl.len().alias("n_lines"),
    )
    .sort("revenue", descending=True)
    .head(10)
)

%time cpu = q3.collect(engine="streaming")
%time gpu = q3.collect(engine=engine)
gpu


CPU times: user 14min 55s, sys: 2min 41s, total: 17min 37s
Wall time: 38.2 s
CPU times: user 176 ms, sys: 4.71 s, total: 4.89 s
Wall time: 17.2 s


l_partkey,l_suppkey,revenue,n_lines
i64,i64,f64,u64
140566994,566995,1.3011e6,23
28517954,1017957,1.2987e6,23
118942915,6442949,1.2983e6,22
180206994,7707049,1.2879e6,22
52312909,7312920,1.2822e6,23
28793951,1293954,1.2816e6,22
134191994,1692034,1.2767e6,22
179786991,4787026,1.2676e6,22
78370935,8370936,1.2630e6,21


The CPU spent 17.6 minutes of processor time, and the GPU 4.9 seconds.

# Q4. Which brand wins?
Brand isn’t stored with each shipment, so we look up every one of the 2.5 billion lines in the 200-million-part catalog. The winner is Brand No.43 at 3.61 trillion. Last place is Brand No.11 at 3.6055 trillion, only 0.19% behind. It’s a photo finish among 25 brands. 

In [9]:
# Query 4 (hardest): join lineitem (billions of rows) to part (200M rows), revenue per brand
part = pl.scan_parquet("s3://rapids-tpch/scale-1000/part/*.parquet")

q4 = (
    lineitem
    .join(part.select("p_partkey", "p_brand"), left_on="l_partkey", right_on="p_partkey")
    .group_by("p_brand")
    .agg((pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).sum().alias("revenue"))
    .sort("revenue", descending=True)
)

%time cpu = q4.collect()
%time gpu = q4.collect(engine=engine)
gpu


CPU times: user 12min 51s, sys: 2min 24s, total: 15min 15s
Wall time: 38.3 s
CPU times: user 170 ms, sys: 12.2 s, total: 12.3 s
Wall time: 16 s


p_brand,revenue
str,f64
"""Brand#43""",3.6122e12
"""Brand#45""",3.6115e12
"""Brand#35""",3.6106e12
"""Brand#54""",3.6104e12
"""Brand#52""",3.6102e12
…,…
"""Brand#23""",3.6085e12
"""Brand#22""",3.6084e12
"""Brand#41""",3.6080e12


This is the hardest query and the biggest gap, 2.4x.

Four very different questions, and each one finished in under 22 seconds on the GPU. On the CPU, the same answers took about 38 seconds and 12 to 18 minutes of processor time.

In [10]:
# Done: stop the GPU workers
engine.shutdown()
